# Coarse-grained protein folding: a Gō model with Langevin dynamics

*biophysics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/biophysics/coarse_grained_md)

A small helical-hairpin protein as a chain of beads, one per amino acid, held together by bonds and bond angles and stabilised by attractive native contacts (a Gō model). Forces are the autograd gradient of the total energy, written with pairwise-distance tensors. Langevin dynamics runs a whole batch of temperatures and replicas at once, half started folded and half unfolded; the fraction of native contacts $Q(t)$ shows folding and unfolding, and its average against temperature is the melting curve.

**Vector form**

$$m\,\ddot{\mathbf x}_i = -\nabla_{\mathbf x_i} E - \gamma m\,\dot{\mathbf x}_i + \sqrt{2\gamma m k_B T}\;\boldsymbol\xi_i(t),\qquad E = \sum_{\text{bonds}} k_b (r - b)^2 + \sum_{\text{angles}} k_\theta (\theta - \theta^0)^2 + \sum_{\text{native}} \varepsilon\Big[5\Big(\frac{r^0}{r}\Big)^{12} - 6\Big(\frac{r^0}{r}\Big)^{10}\Big] + \sum_{\text{other}} \varepsilon\Big(\frac{\sigma}{r}\Big)^{12}$$

**Index (tensor) form**

$$r_{sp} = \sqrt{(x_{s i_p a} - x_{s j_p a})(x_{s i_p a} - x_{s j_p a})} \qquad E_s = \dots + n_p\,\varepsilon\big[5 (r^0_p/r_{sp})^{12} - 6 (r^0_p/r_{sp})^{10}\big] + (1 - n_p)\,\varepsilon\,(\sigma/r_{sp})^{12} \qquad F_{sia} = -\frac{\partial E_s}{\partial x_{sia}} \qquad Q_s = \frac{1}{M}\,n_p\,\big[r_{sp} < 1.2\,r^0_p\big]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{sia}$ | `initial_chains(native, B, R)` | (S, N, 3) | bead $i$ of system $s$; $S = B\cdot R$ temperatures × replicas |
| $r^0_p,\ n_p$ | `go_model(native)['r0'], ['native']` | (P,) | native distance of pair $p = (i_p, j_p)$, $j - i \ge 3$; $n_p = 1$ if closer than 8 Å in the native fold |
| $r_{sp}$ | `pair_list_distances(x, i, j)` | (S, P) | all pair distances of every system in one tensor, $a$ summed |
| $E_s$ | `total_energy(x, model)` | (S,) | bonds + angles + 12–10 native wells (minimum $-\varepsilon$ at $r^0_p$) + repulsion of non-native pairs |
| $F_{sia} = -\partial E_s/\partial x_{sia}$ | `torch.autograd.grad(E.sum(), x)` | (S, N, 3) | forces by automatic differentiation; summing over $s$ is fine because the systems do not interact |
| $T_s$ | `temperatures(T_min, T_max, B).repeat_interleave(R)` | (S,) | each system gets its own heat bath |
| $\text{BAOAB}$ | `langevin(x0, T, model, dt, gamma, n_steps)` | (K, S) | kick, drift, friction + noise $v \to e^{-\gamma dt}v + \sqrt{(1-e^{-2\gamma dt})k_BT}\,\xi$, drift, kick |
| $Q_s$ | `fraction_native(x, model)` | (S,) | share of native contacts formed: 1 folded, about 0.4 for an unfolded chain (local contacts keep forming) |
| $\langle Q\rangle(T)$ | `melting_curve(times, Q, B, R)` | (B,) | time average over the second half of the run; $T_m$ where it crosses ½ |

A Gō model is built from the answer: every interaction that exists in the native structure is attractive and everything else only repels, so the energy landscape is a smooth funnel towards the native fold. That is a caricature, but a useful one: small proteins really do fold as if their landscape were minimally frustrated, and Gō models reproduce folding rates, transition states and the cooperative melting of many of them. Units are reduced: energies in $\varepsilon$ (one contact), temperatures in $\varepsilon/k_B$, time in $\tau$. Half of the replicas start folded and half as a straight rod; where the solid and dashed curves agree the runs have equilibrated, where they do not (at low $T$) the unfolded chains are kinetically trapped in misfolded states and would need much longer runs or replica exchange. The melting curve is broad because the protein is tiny; real proteins with hundreds of contacts melt sharply. Without a dihedral term the energy depends only on distances, so the mirror image of the native fold has exactly the same energy and $Q$; real Gō models add dihedral potentials to fix the handedness. Try changing the friction (it changes the kinetics but not the equilibrium curve) or the number of temperatures.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/biophysics/coarse_grained_md/main.py
"""Main loop: native fold → Gō model → Langevin runs at many temperatures → Q(t) and the melting curve."""

from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import total_energy


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    native = tensors.native_structure(params["helix_length"], dev)  # (N, 3) Å
    model = tensors.go_model(native)
    n_temps, n_rep = params["temperatures"], 4  # 2 replicas start folded, 2 unfolded
    T_b = tensors.temperatures(params["T_min"], params["T_max"], n_temps, dev)  # (B,)
    T = T_b.repeat_interleave(n_rep)  # (S,) = (B·R,)
    x0 = tensors.initial_chains(native, n_temps, n_rep)  # (S, N, 3)
    print(
        f"Gō model: {native.shape[0]} beads, {int(model['native'].sum())} native contacts, "
        f"native energy {total_energy(native[None], model).item():.1f} ε"
    )
    print(f"{n_temps} temperatures × {n_rep} replicas = {len(T)} chains integrated together")

    n_steps = 600 if fast_mode() else params["steps"]
    dt = 0.02  # τ; the stiff bonds (period ≈ 0.44 τ) set the limit
    times, Q, frame_t, frames = solve.langevin(x0, T, model, dt, params["friction"], n_steps)
    q_nat, q_unf, q_all = solve.melting_curve(times, Q, n_temps, n_rep)
    print(f"simulated {n_steps} steps × dt {dt} = {n_steps * dt:.0f} τ")
    print("   T     ⟨Q⟩ from folded   ⟨Q⟩ from unfolded")
    for b in range(n_temps):
        print(f"  {T_b[b].item():.2f}      {q_nat[b].item():.2f}             {q_unf[b].item():.2f}")
    Tm = solve.melting_temperature(T_b, q_all)
    print(f"melting temperature (⟨Q⟩ = ½): T_m ≈ {Tm:.2f} ε/k_B")
    return plot.animate(native, times, Q, frame_t, frames, T_b, n_rep, Tm, out_dir / "coarse_grained_md.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/biophysics/coarse_grained_md/equation.py
r"""Physics → tensor translation: the Gō-model energy of a bead chain, and forces from autograd.

Indices: s system (temperature × replica), i bead, p pair (i_p, j_p) with j - i ≥ 3, a Cartesian axis.

    r_sp = √((x_{s i_p a} - x_{s j_p a})(x_{s i_p a} - x_{s j_p a}))      (a summed)
    E_s = Σ_i k_b (|x_{s,i+1} - x_{si}| - b_i)²                            bonds
        + Σ_i k_θ (θ_si - θ⁰_i)²                                           bond angles
        + Σ_p n_p ε [5 (r⁰_p / r_sp)¹² - 6 (r⁰_p / r_sp)¹⁰]                 native contacts (n_p = 1), min. -ε at r⁰_p
        + Σ_p (1 - n_p) ε (σ / r_sp)¹²                                     every other pair only repels
    F_sia = -∂E_s / ∂x_sia                                                 (torch.autograd)
    Q_s = (1 / M) Σ_p n_p [r_sp < 1.2 r⁰_p]                                fraction of native contacts
"""

import torch

K_BOND = 100.0  # ε / Å²
K_ANGLE = 20.0  # ε / rad²
EPS = 1.0  # ε, depth of one native contact
SIGMA_REP = 4.0  # Å, size of the non-native repulsion


def pair_distances(x: torch.Tensor) -> torch.Tensor:
    """Full distance matrix r_ij for coordinates (..., N, 3) → (..., N, N); the axis index a is summed."""
    diff = x[..., :, None, :] - x[..., None, :, :]
    return torch.sqrt(torch.einsum("...ija,...ija->...ij", diff, diff) + 1e-9)


def pair_list_distances(x: torch.Tensor, i: torch.Tensor, j: torch.Tensor) -> torch.Tensor:
    """r_sp = |x_{s,i_p} - x_{s,j_p}| for the listed pairs only.   x (S, N, 3), i, j (P,) → (S, P).

    The a-sum has just 3 terms, so it is (diff * diff).sum(-1) rather than an einsum: this runs every time step,
    and a few larger tensor operations keep a small simulation fast on a CPU.
    """
    diff = x[:, i] - x[:, j]  # (S, P, 3)
    return torch.sqrt((diff * diff).sum(-1))


def bond_energy(x: torch.Tensor, b0: torch.Tensor) -> torch.Tensor:
    """Σ_i k_b (|x_{i+1} - x_i| - b_i)²  for x (S, N, 3).   (S,)"""
    length = (x[:, 1:] - x[:, :-1]).norm(dim=-1)  # (S, N-1)
    return K_BOND * ((length - b0) ** 2).sum(-1)


def angle_energy(x: torch.Tensor, theta0: torch.Tensor) -> torch.Tensor:
    """Σ_i k_θ (θ_i - θ⁰_i)², θ_i the angle at bead i+1 between its two bonds.   (S,)"""
    b = x[:, 1:] - x[:, :-1]  # (S, N-1, 3) bond vectors
    cos = -(b[:, :-1] * b[:, 1:]).sum(-1) / (b[:, :-1].norm(dim=-1) * b[:, 1:].norm(dim=-1))  # a summed
    theta = torch.arccos(cos.clamp(-0.9999, 0.9999))
    return K_ANGLE * ((theta - theta0) ** 2).sum(-1)


def pair_energy(r: torch.Tensor, r0: torch.Tensor, native: torch.Tensor) -> torch.Tensor:
    """Native pairs: ε [5 (r⁰/r)¹² - 6 (r⁰/r)¹⁰], a 12–10 well with its minimum -ε exactly at r⁰.
    Other pairs: ε (σ/r)¹², pure repulsion.   r (S, P), r0 (P,), native (P,) as 0/1 floats → (S,)."""
    q2 = (r0 / r) ** 2
    q10 = q2**5
    well = EPS * (5 * q10 * q2 - 6 * q10)
    wall = EPS * ((SIGMA_REP / r) ** 2) ** 6
    return (native * well + (1 - native) * wall).sum(-1)


def total_energy(x: torch.Tensor, model: dict) -> torch.Tensor:
    """E_s for every system at once, x (S, N, 3) → (S,)."""
    r = pair_list_distances(x, model["i"], model["j"])  # (S, P)
    return bond_energy(x, model["b0"]) + angle_energy(x, model["theta0"]) + pair_energy(r, model["r0"], model["native"])


def forces(x: torch.Tensor, model: dict) -> torch.Tensor:
    """F_sia = -∂E_s/∂x_sia by automatic differentiation.   (S, N, 3)

    Summing E over systems is safe: system s's energy depends only on its own coordinates.
    """
    with torch.enable_grad():
        x = x.detach().requires_grad_(True)
        (grad,) = torch.autograd.grad(total_energy(x, model).sum(), x)
    return -grad


def fraction_native(x: torch.Tensor, model: dict) -> torch.Tensor:
    """Q_s: share of native contacts within 1.2 × their native distance.   (S,)"""
    r = pair_list_distances(x, model["i"], model["j"])
    formed = (r < 1.2 * model["r0"]).float() * model["native"]
    return formed.sum(-1) / model["native"].sum()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/biophysics/coarse_grained_md/tensors.py
"""Build the concrete tensors: the native fold, the Gō-model parameters read off it, and the starting chains.

Units: lengths in Å, energies in ε (the depth of one native contact), mass 1, temperature in ε / k_B,
time in τ = Å √(m / ε).
"""

import math

import torch

from .equation import pair_distances

CONTACT_CUTOFF = 8.0  # Å: residues closer than this in the native fold form a native contact
MIN_SEQ_SEP = 3  # only pairs |i - j| ≥ 3 count as contacts (closer ones are held by bonds and angles)


def native_structure(n_helix: int, device: torch.device) -> torch.Tensor:
    """Cα beads of a small helical hairpin, shape (N, 3): two antiparallel right-handed helices and a 3-bead loop."""
    r, rise, turn, sep = 2.3, 1.5, math.radians(100.0), 8.5  # α-helix geometry, Å; 8.5 Å between the helix axes
    t = torch.arange(n_helix, dtype=torch.float32)
    phi = turn * t
    h1 = torch.stack([-sep / 2 + r * torch.cos(phi), r * torch.sin(phi), rise * t], -1)
    top = rise * (n_helix - 1)
    h2 = torch.stack([sep / 2 - r * torch.cos(phi), r * torch.sin(phi), top - rise * t], -1)
    # loop: straight line between the helix ends, lifted into an arch
    s = torch.linspace(0, 1, 5)[1:-1, None]  # 3 interior points
    loop = (1 - s) * h1[-1] + s * h2[0] + torch.tensor([0.0, 0.0, 3.5]) * torch.sin(math.pi * s)
    return torch.cat([h1, loop, h2]).to(device)


def go_model(native: torch.Tensor) -> dict:
    """Gō model: every bond length, bond angle and contact distance is set to its value in the native fold.

    Returns b0 (N-1,) bond lengths, theta0 (N-2,) bond angles, and for every pair p = (i_p, j_p) with
    j - i ≥ 3: i, j (P,) indices, r0 (P,) native distance, native (P,) 1.0 for a native contact else 0.0.
    """
    n = native.shape[0]
    r_native = pair_distances(native)  # (N, N)
    i, j = torch.triu_indices(n, n, offset=MIN_SEQ_SEP, device=native.device)  # each pair once
    r0 = r_native[i, j]
    b = native[1:] - native[:-1]  # (N-1, 3) bond vectors
    cos = (-(b[:-1] * b[1:]).sum(-1)) / (b[:-1].norm(dim=-1) * b[1:].norm(dim=-1))
    return {
        "b0": b.norm(dim=-1),
        "theta0": torch.arccos(cos),
        "i": i,
        "j": j,
        "r0": r0,
        "native": (r0 < CONTACT_CUTOFF).to(r0.dtype),
    }


def temperatures(t_min: float, t_max: float, n: int, device: torch.device) -> torch.Tensor:
    """T_b, shape (B,): evenly spaced temperatures, one column of replicas each."""
    return torch.linspace(t_min, t_max, n, device=device)


def initial_chains(native: torch.Tensor, n_temps: int, n_rep: int, seed: int = 0) -> torch.Tensor:
    """x0 of shape (B·R, N, 3): for every temperature, half the replicas start folded (native) and half
    as a straight, fully unfolded chain. Agreement between the two halves means the run has equilibrated."""
    gen = torch.Generator().manual_seed(seed)
    n = native.shape[0]
    bond = (native[1:] - native[:-1]).norm(dim=-1).mean().item()
    straight = torch.zeros(n, 3, device=native.device)
    straight[:, 0] = bond * torch.arange(n, device=native.device)
    starts = [native if r < n_rep // 2 else straight for r in range(n_rep)]
    x0 = torch.stack(starts)[None].expand(n_temps, -1, -1, -1).reshape(-1, n, 3)
    return x0 + 0.05 * torch.randn(x0.shape, generator=gen).to(native.device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/biophysics/coarse_grained_md/solve.py
"""Langevin dynamics (BAOAB splitting) for every temperature and replica at once."""

import math

import torch

from problems._common.progress import steps

from .equation import forces, fraction_native


def langevin(
    x0: torch.Tensor, T: torch.Tensor, model: dict, dt: float, gamma: float, n_steps: int, n_frames: int = 120
):
    """Integrate m dv = F dt - γ m v dt + √(2 γ m k_B T) dW for x0 (S, N, 3), one temperature T_s per system.

    BAOAB: half kick (B), half drift (A), exact friction + noise (O), half drift, half kick. Mass m = 1.
    Returns (times (K,), Q history (K, S), frame times (F,), frames (F, S, N, 3)).
    """
    gen = torch.Generator().manual_seed(1)
    c1 = math.exp(-gamma * dt)  # velocity kept by the friction step
    c2 = torch.sqrt((1 - c1**2) * T)[:, None, None]  # (S, 1, 1): thermal kick size, √(k_B T / m) per component
    x = x0.clone()
    v = torch.randn(x.shape, generator=gen).to(x.device) * torch.sqrt(T)[:, None, None]  # Maxwell–Boltzmann
    f = forces(x, model)
    q_every = max(1, n_steps // 600)
    f_every = max(1, n_steps // n_frames)
    times, qs, frame_t, frames = [0.0], [fraction_native(x, model)], [0.0], [x.clone()]
    for step in steps(n_steps, "Langevin steps"):
        v = v + 0.5 * dt * f  # B
        x = x + 0.5 * dt * v  # A
        v = c1 * v + c2 * torch.randn(v.shape, generator=gen).to(v.device)  # O
        x = x + 0.5 * dt * v  # A
        f = forces(x, model)
        v = v + 0.5 * dt * f  # B
        t = (step + 1) * dt
        if (step + 1) % q_every == 0:
            times.append(t)
            qs.append(fraction_native(x, model))
        if (step + 1) % f_every == 0:
            frame_t.append(t)
            frames.append(x.clone())
    return torch.tensor(times), torch.stack(qs), torch.tensor(frame_t), torch.stack(frames)


def melting_curve(times: torch.Tensor, Q: torch.Tensor, n_temps: int, n_rep: int, t_now: float | None = None):
    """⟨Q⟩ per temperature over the second half of the run (up to t_now), split by start.

    Q: (K, B·R). Returns (from native (B,), from unfolded (B,), all (B,)).
    """
    t_now = times[-1].item() if t_now is None else t_now
    window = (times >= 0.5 * t_now) & (times <= t_now)
    q = Q[window].reshape(-1, n_temps, n_rep).mean(0)  # (B, R)
    half = n_rep // 2
    return q[:, :half].mean(1), q[:, half:].mean(1), q.mean(1)


def melting_temperature(T: torch.Tensor, q: torch.Tensor) -> float:
    """Where ⟨Q⟩ first drops through ½, by linear interpolation (nan if it never does)."""
    for b in range(len(T) - 1):
        if q[b] >= 0.5 > q[b + 1]:
            return (T[b] + (q[b] - 0.5) / (q[b] - q[b + 1]) * (T[b + 1] - T[b])).item()
    return float("nan")


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/biophysics/coarse_grained_md/plot.py
"""Animate the runs: chains at three temperatures, Q(t) for every temperature, and the melting curve building up."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .solve import melting_curve

SPACING = 26.0  # Å between the three chains drawn side by side


def animate(
    native: torch.Tensor,
    times: torch.Tensor,
    Q: torch.Tensor,
    frame_t: torch.Tensor,
    frames: torch.Tensor,
    T_b: torch.Tensor,
    n_rep: int,
    Tm: float,
    path: Path,
) -> Path:
    """native (N, 3); Q (K, S) at times (K,); frames (F, S, N, 3) at frame_t (F,); S = B·R systems."""
    nat = native.cpu().numpy()
    nat = nat - nat.mean(0)
    t, q = times.cpu().numpy(), Q.cpu().numpy()
    ft, fr = frame_t.cpu().numpy(), frames.cpu().numpy()
    temps = T_b.cpu().numpy()
    B, half = len(temps), n_rep // 2
    colours = plt.get_cmap("coolwarm")(np.linspace(0, 1, B))
    q_br = q.reshape(len(t), B, n_rep)
    q_nat = _smooth(q_br[:, :, :half].mean(2))  # (K, B) started folded
    q_unf = _smooth(q_br[:, :, half:].mean(2))  # (K, B) started unfolded
    final_nat, final_unf, _ = (c.cpu().numpy() for c in melting_curve(times, Q, B, n_rep))

    # three chains that started unfolded: coldest, nearest T_m, hottest
    near_tm = int(np.nanargmin(np.abs(temps - Tm))) if np.isfinite(Tm) else B // 2
    shown = sorted({0, near_tm, B - 1})
    systems = [b * n_rep + half for b in shown]

    fig = plt.figure(figsize=(10.5, 4.0))
    grid = fig.add_gridspec(1, 3, width_ratios=[1.35, 1, 1])
    ax3 = fig.add_subplot(grid[0], projection="3d")
    ax_q = fig.add_subplot(grid[1])
    ax_m = fig.add_subplot(grid[2])

    offsets = [np.array([(k - (len(shown) - 1) / 2) * SPACING, 0.0, 0.0]) for k in range(len(shown))]
    chains = []
    for k, b in enumerate(shown):
        ax3.plot(*(nat + offsets[k]).T, color="#d1d5db", lw=1)
        chains.append(ax3.plot([], [], [], "o-", color=colours[b], ms=3, lw=1.5)[0])
        ax3.text(*(offsets[k] + [0, 0, 15]), f"T = {temps[b]:.2f}", ha="center", fontsize=8)
    lim = SPACING * len(shown) / 2
    ax3.set_xlim(-lim, lim)
    ax3.set_ylim(-13, 13)
    ax3.set_zlim(-13, 13)
    ax3.set_box_aspect((2 * lim, 26, 26), zoom=1.5)
    ax3.set_axis_off()
    title3 = ax3.set_title("", fontsize=10)

    ax_q.set_xlim(0, t[-1])
    ax_q.set_ylim(0, 1.02)
    ax_q.set_xlabel("time (τ)")
    ax_q.set_ylabel("fraction of native contacts Q")
    ax_q.set_title("solid: started folded, dashed: unfolded", fontsize=9)
    lines_n = [ax_q.plot([], [], color=colours[b], lw=1.2)[0] for b in range(B)]
    lines_u = [ax_q.plot([], [], "--", color=colours[b], lw=1.2)[0] for b in range(B)]

    ax_m.plot(temps, final_nat, color="#9ca3af", lw=1)
    ax_m.plot(temps, final_unf, "--", color="#9ca3af", lw=1)
    if np.isfinite(Tm):
        ax_m.axvline(Tm, color="#f59e0b", lw=1, ls=":")
        ax_m.text(Tm, 0.04, f" Tm ≈ {Tm:.2f}", color="#b45309", fontsize=8)
    pts_n = ax_m.scatter(temps, np.zeros(B), c=colours, s=28, edgecolors="#374151", zorder=3)
    pts_u = ax_m.scatter(temps, np.zeros(B), c=colours, s=28, marker="s", edgecolors="#374151", zorder=3)
    ax_m.set_xlim(temps[0] - 0.05, temps[-1] + 0.05)
    ax_m.set_ylim(0, 1.02)
    ax_m.set_xlabel("temperature T (ε / k_B)")
    ax_m.set_ylabel("⟨Q⟩, second half of the run so far")
    ax_m.set_title("melting curve (● folded, ■ unfolded start)", fontsize=9)

    native_t = torch.as_tensor(nat)

    def update(f: int):
        now = ft[f]
        k = int(np.searchsorted(t, now, side="right"))
        for b in range(B):
            lines_n[b].set_data(t[:k], q_nat[:k, b])
            lines_u[b].set_data(t[:k], q_unf[:k, b])
        for c, s in enumerate(systems):
            x = _superpose(fr[f, s], native_t) + offsets[c]
            chains[c].set_data_3d(x[:, 0], x[:, 1], x[:, 2])
        if now > 0:
            m_n, m_u, _ = melting_curve(times, Q, B, n_rep, float(now))
            pts_n.set_offsets(np.c_[temps, m_n.cpu().numpy()])
            pts_u.set_offsets(np.c_[temps, m_u.cpu().numpy()])
        ax3.view_init(elev=15, azim=-80 + 40 * f / max(len(ft) - 1, 1))
        title3.set_text(f"chains started unfolded, t = {now:.0f} τ")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(ft), 72), path, fps=8)


def _smooth(y: np.ndarray, width: int = 15) -> np.ndarray:
    """Running mean along time (axis 0), so the thermal jitter of Q does not hide the trend."""
    kernel = np.ones(width) / width
    pad = np.pad(y, ((width // 2, width - 1 - width // 2), (0, 0)), mode="edge")
    return np.stack([np.convolve(pad[:, b], kernel, mode="valid") for b in range(y.shape[1])], 1)


def _superpose(x: np.ndarray, ref: torch.Tensor) -> np.ndarray:
    """Kabsch: rotate the centred chain x (N, 3) onto the centred reference, for display only."""
    P = torch.as_tensor(x - x.mean(0))
    H = P.T @ ref  # (3, 3) = Σ_i p_ia q_ib
    U, _, Vh = torch.linalg.svd(H)
    d = torch.sign(torch.linalg.det(Vh.T @ U.T))
    R = Vh.T @ torch.diag(torch.tensor([1.0, 1.0, d.item()], dtype=P.dtype)) @ U.T
    return (P @ R.T).numpy()


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'helix_length': 8,
    'T_min': 0.3,
    'T_max': 1.3,
    'temperatures': 8,
    'friction': 0.3,
    'steps': 8000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.biophysics.coarse_grained_md")]:
    del sys.modules[name]
main = importlib.import_module("problems.biophysics.coarse_grained_md.main")
problem_dir = Path("problems/biophysics/coarse_grained_md")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
